[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/02_Embeddings_and_RAG/embeddings_and_rag.ipynb)

# Module 02: Embeddings & Retrieval-Augmented Generation (RAG)

This notebook takes a first-principles, build-from-scratch approach to understanding embeddings and RAG. We use only Vanilla Python, NumPy, and Pydantic—no high-level frameworks.

## 1. Foundations of Embeddings

Embeddings map real-world data (like text) into a low-dimensional numerical vector space. This process acts as a form of lossy compression, retaining semantic meaning while enabling efficient computation.

Mathematically, an embedding is a function $f: X \rightarrow \mathbb{R}^d$ where $X$ is the input space and $d$ is the embedding dimension.

### Why Embeddings Matter

Embeddings allow us to compare the meaning of different pieces of data using geometric distance. For example, similar sentences will have vectors that are close together in the embedding space.

In [ ]:
# Install dependencies (uncomment if running in Colab)
# !pip install numpy pydantic

import numpy as np
from pydantic import BaseModel, Field
from typing import List, Optional

### Manual Cosine Similarity

Cosine similarity is a common metric for measuring the similarity between two vectors. It is defined as:

$$
\text{cosine}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}
$$

This value ranges from -1 (opposite) to 1 (identical), with 0 indicating orthogonality (no similarity).

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    Compute the cosine similarity between two vectors.
    Args:
        a: First vector (1D numpy array)
        b: Second vector (1D numpy array)
    Returns:
        Cosine similarity (float)
    """
    # Defensive: Ensure no division by zero
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Example usage:
vec1 = np.array([1, 2, 3])
vec2 = np.array([4, 5, 6])
print(f"Cosine similarity: {cosine_similarity(vec1, vec2):.4f}")

## 2. Vector Storage with Pydantic

To manage document IDs, metadata, and embeddings with strict type validation, we use Pydantic models. This ensures our vector store is robust and easy to debug.

In [ ]:
class VectorRecord(BaseModel):
    doc_id: str
    embedding: List[float]
    metadata: Optional[dict] = None

class SimpleVectorStore(BaseModel):
    records: List[VectorRecord] = []

    def add(self, record: VectorRecord):
        # Add a new vector record to the store
        self.records.append(record)

    def search(self, query_vec: np.ndarray, top_k: int = 1):
        # Return the top_k most similar records to the query vector
        scored = [
            (rec, cosine_similarity(np.array(rec.embedding), query_vec))
            for rec in self.records
        ]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

# Example usage:
store = SimpleVectorStore()
store.add(VectorRecord(doc_id="doc1", embedding=[0.1, 0.2, 0.3], metadata={"title": "Doc 1"}))
store.add(VectorRecord(doc_id="doc2", embedding=[0.4, 0.5, 0.6], metadata={"title": "Doc 2"}))
query = np.array([0.1, 0.2, 0.3])
results = store.search(query, top_k=2)
for rec, score in results:
    print(f"Found: {rec.doc_id} (score={score:.4f})")

## 3. The RAG Architectural Loop

Retrieval-Augmented Generation (RAG) combines two phases:

- **Precomputing Phase:** Index documents by embedding them and storing in a vector store.
- **Retrieval Phase:** Map a query to the same vector space, retrieve the most similar documents, and inject their content into the prompt for the LLM.

This reduces hallucinations by grounding the model in real data.

In [ ]:
# Example: Precomputing Phase (Indexing)
documents = [
    {"id": "doc1", "text": "The capital of France is Paris."},
    {"id": "doc2", "text": "The tallest mountain is Mount Everest."},
    {"id": "doc3", "text": "Python is a popular programming language."}
]

def simple_text_embedding(text: str) -> np.ndarray:
    """
    Simple embedding: sum of character codes, padded/truncated to length 8.
    This is for demonstration only; real embeddings use ML models.
    """
    arr = np.array([ord(c) for c in text if c.isalnum()])
    arr = arr[:8] if len(arr) >= 8 else np.pad(arr, (0, 8 - len(arr)), 'constant')
    return arr.astype(float)

store = SimpleVectorStore(records=[])
for doc in documents:
    emb = simple_text_embedding(doc["text"])
    store.add(VectorRecord(doc_id=doc["id"], embedding=emb.tolist(), metadata={"text": doc["text"]}))

# Example: Retrieval Phase
query = "What is the capital of France?"
query_emb = simple_text_embedding(query)
results = store.search(query_emb, top_k=1)
retrieved_text = results[0][0].metadata["text"]
print(f"Retrieved: {retrieved_text}")

### Augmentation Step: Reducing Hallucinations

Inject the retrieved text into the system prompt for the LLM. This grounds the model's response in real data.

Example prompt:

> System: You are a helpful assistant. Use the following context to answer the question.
>
> Context: The capital of France is Paris.
>
> Question: What is the capital of France?

In [ ]:
# Example: Constructing the Augmented Prompt
system_prompt = "You are a helpful assistant. Use the following context to answer the question."
context = retrieved_text
question = query
augmented_prompt = f"{system_prompt}\n\nContext: {context}\n\nQuestion: {question}"
print(augmented_prompt)

## 4. Scaling Concepts: ANN Algorithms

For small datasets, brute-force search is feasible. But as the number of documents grows, we need faster methods.

**Approximate Nearest Neighbor (ANN)** algorithms like ScaNN and HNSW allow us to find similar vectors quickly, trading a small amount of accuracy for massive speed gains.

- **ScaNN:** Uses partitioning and quantization for efficient search.
- **HNSW:** Builds a navigable small-world graph for fast traversal.

These are essential for production-scale retrieval, but the core logic remains the same: find the closest vectors in embedding space.

## 5. Hands-on Challenge: Build and Query Your Own Knowledge Base

**Challenge:**
1. Populate the vector store with at least 3 new facts (use your own sentences).
2. Write a query that should retrieve one of your facts.
3. Print the retrieved result and the constructed augmented prompt.

*This exercise will help you understand the full RAG loop, from indexing to retrieval to prompt augmentation.*

In [ ]:
# TODO: Complete the challenge below
# 1. Add at least 3 new facts to the vector store
# 2. Write a query that should retrieve one of your facts
# 3. Print the retrieved result and the constructed augmented prompt

# Example template:
my_facts = [
    {"id": "fact1", "text": "The Pacific Ocean is the largest ocean on Earth."},
    {"id": "fact2", "text": "The speed of light is approximately 299,792 kilometers per second."},
    {"id": "fact3", "text": "The Mona Lisa was painted by Leonardo da Vinci."}
]

my_store = SimpleVectorStore(records=[])
for fact in my_facts:
    emb = simple_text_embedding(fact["text"])
    my_store.add(VectorRecord(doc_id=fact["id"], embedding=emb.tolist(), metadata={"text": fact["text"]}))

my_query = "Who painted the Mona Lisa?"
my_query_emb = simple_text_embedding(my_query)
my_results = my_store.search(my_query_emb, top_k=1)
my_retrieved_text = my_results[0][0].metadata["text"]

my_system_prompt = "You are a helpful assistant. Use the following context to answer the question."
my_augmented_prompt = f"{my_system_prompt}\n\nContext: {my_retrieved_text}\n\nQuestion: {my_query}"

print(f"Retrieved: {my_retrieved_text}")
print("\nAugmented Prompt:\n", my_augmented_prompt)